In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 6b: the Gale air-temperature map

Air = `air_hybrid` (`residual.AIR_VARIANTS`), refit on all fitting rows with each row's out-of-fold predicted ground (`outputs/model_scores/5_air_model/air_rows.csv`), then applied frame by frame to the ground map masters (`outputs/maps/ground/_working/master/ls{LLL}.npy`). No solver run. Design: `docs/plan.md`; scores: `notebooks/air_lobo.ipynb`.

**Status of the model:** `air_hybrid` fails the adoption rule (LOBO macro 4.05 K vs the benchmark's 4.04 K, better in 4 of 9 blocks). It is published under the fallback fixed, as the better of the two configurations (`air_time_only` 4.35 K). User go-ahead with that caveat: .

- **`MODE = "dry"`**: the 100 x 110 rover window (exported in all four formats and checked) and every 20th pixel across the crater (for the colour range and the extrapolation fraction). Output in `outputs/maps/air/dryrun/`.
- **`MODE = "full"`**: the whole 718 x 738 grid, output in `outputs/maps/air/`.

In [ ]:
# --- config ---
import json
import subprocess
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r
import map_export as m

MODE = "full"                 # "dry" or "full"
AIR_COLOR_RANGE_K = m.COLOR_RANGE_K   # : the ground scale, so colours compare (dry-run rule gave 190-280 K); None in a dry run
COLOR_ROUND_K = 10.0          # dry-run rule: crater-sample min and max, rounded outward to 10 K
ROVER_200M = (152, 186)
REPRO_TOL_K = 1e-3            # map path vs direct model prediction at the rover pixel (float32 rounding)
N_FIT_ROWS, N_BASIS = 12156, 24   # air_lobo.ipynb, all training blocks
LOBO = {"air_hybrid_macro_k": 4.052, "benchmark_macro_k": 4.037, "air_time_only_macro_k": 4.350,
        "blocks_better_than_benchmark": 4, "air_hybrid_macro_no_block0_k": 3.83, "benchmark_macro_no_block0_k": 4.05}
COMPARE_LS_WINDOW_DEG = 15.0  # REMS sols within this Ls of each map Ls, for the rover-pixel comparison
SPREAD_HOURS = (2.0, 14.0)
DRY_RUNS = {"rover": ((102, 202, 1, 131, 241, 1), True), "sample": ((5, 718, 20, 5, 738, 20), False)}
FULL_RUNS = {"full": (m.FULL_WINDOW, True)}
META = paths.DATA_DIR / "gale_dust_devil_meta.json"
TABLE = paths.DATA_DIR / "gale_rems_training.csv"
AIR_ROWS = paths.AIR_OUT_DIR / "air_rows.csv"
GROUND_RANGE = paths.AIR_OUT_DIR / "air_ground_pred_range.csv"
GROUND_DIR = paths.MAP_OUT_DIR
GROUND_MANIFEST = GROUND_DIR / "map_manifest.json"

assert MODE in ("dry", "full"), MODE
assert (MODE == "dry") == (AIR_COLOR_RANGE_K is None), "set AIR_COLOR_RANGE_K from the dry run before the full run"
RUNS = DRY_RUNS if MODE == "dry" else FULL_RUNS
OUT = paths.AIR_MAP_OUT_DIR / ("dryrun" if MODE == "dry" else "")
OUT.mkdir(parents=True, exist_ok=True)
run_dir = {name: OUT / name if MODE == "dry" else OUT for name in RUNS}
print("mode:", MODE, "| out:", OUT, "| air model:", m.AIR_VARIANT, r.AIR_VARIANTS[m.AIR_VARIANT])

## Inputs: scenarios from the ground map, static grid, air model

In [ ]:
gman = json.loads(GROUND_MANIFEST.read_text())
assert gman["mode"] == "full", "ground manifest is not from the full run"
LS_TAU = [(s["ls_deg"], s["tau"]) for s in gman["scenarios"]]
assert [ls for ls, _ in LS_TAU] == list(m.MAP_LS_DEG)
print("scenarios (from the ground manifest):", [(ls, round(tau, 3)) for ls, tau in LS_TAU])
meta = m.load_meta(META)
static = dict(np.load(GROUND_DIR / m.WORK_SUBDIR / "static_200m.npz"))
transform_full = m.map_transform(meta)
assert np.allclose(transform_full, gman["grid"]["transform_gdal"], rtol=0, atol=1e-12), "transform differs from the ground map"
rover_lonlat = (meta["rover_lon"], meta["rover_lat"])
assert m.lonlat_to_rowcol(transform_full, *rover_lonlat) == ROVER_200M

tbl = pd.DataFrame(t.load_rems_table(TABLE))
air_rows = pd.read_csv(AIR_ROWS)
fit_frame, fit_target = m.air_training_rows(tbl, air_rows)
model = r.FlannModel(r.AIR_VARIANTS[m.AIR_VARIANT], False, fit_frame, fit_target)
n_basis = model.pipe[-1].coef_.size
print(f"{m.AIR_VARIANT}: {len(fit_frame)} fitting rows, {n_basis} basis columns")
assert (len(fit_frame), n_basis) == (N_FIT_ROWS, N_BASIS), "refit differs from air_lobo.ipynb"
print(f"in-sample RMSE {np.sqrt(np.mean((model.predict(fit_frame)[0] - fit_target) ** 2)):.2f} K "
      f"(in-sample; the LOBO macro is {LOBO['air_hybrid_macro_k']} K)")
gp_range = pd.read_csv(GROUND_RANGE, index_col=0).iloc[:, 0]
GP_LO, GP_HI = float(gp_range["ground_pred_p01_k"]), float(gp_range["ground_pred_p99_k"])
print(f"rover range of ground_pred (1st-99th percentile): {GP_LO:.1f}-{GP_HI:.1f} K")

## Check: the map path reproduces the model at the rover pixel

For each Ls, the ground map at the rover's 200 m pixel goes through `map_export.add_air`. The same inputs go through `FlannModel.predict` on a frame built as in `air_lobo.ipynb` (the rover's own static columns, which the air basis does not read). Pass: within 0.001 K at all 48 frames.

In [ ]:
repro = []
for ls, tau in LS_TAU:
    g = np.load(GROUND_DIR / m.WORK_SUBDIR / "master" / f"{m.ls_tag(ls)}.npy", mmap_mode="r")[:, ROVER_200M[0], ROVER_200M[1]]
    g = np.asarray(g, dtype=np.float32)[:, None]
    a_map = m.add_air(g, ls, tau, model)[:, 0].astype(np.float64)
    row = tbl.iloc[0]
    direct = pd.DataFrame({"ls_deg": ls, "tau": tau, "ltst_hr": m.FRAME_HOURS, "baseline_k": g[:, 0].astype(np.float64),
                           "ti_themis_tiu": row.ti_themis_tiu, "slope_deg": row.slope_deg, "aspect_deg": row.aspect_deg,
                           "albedo_mean": row.albedo_mean, "elev_dem_m": row.elev_dem_m, "q_abs_wm2": 0.0})
    a_dir = model.predict(direct)[0]
    repro.append({"ls_deg": ls, "max_abs_diff_k": np.abs(a_map - a_dir).max(), "air_min_k": a_map.min(), "air_max_k": a_map.max()})
repro = pd.DataFrame(repro).set_index("ls_deg")
print(repro.to_string())
assert repro.max_abs_diff_k.max() < REPRO_TOL_K, "map path differs from the model"
print(f"PASS: max |map path - model| = {repro.max_abs_diff_k.max():.2e} K")

## Air master arrays, statistics and exports

In [ ]:
def run_window(name, window, export, color_range):
    """Air masters, statistics and (if export) the four formats for one window. Returns a DataFrame by Ls."""
    d = run_dir[name]
    (d / m.WORK_SUBDIR / "master").mkdir(parents=True, exist_ok=True)
    r0, r1, rs, c0, c1, cs = window
    _, valid = m.window_inputs(static, window)
    rows = []
    for ls, tau in LS_TAU:
        tag = m.ls_tag(ls)
        t0 = time.time()
        G = np.asarray(np.load(GROUND_DIR / m.WORK_SUBDIR / "master" / f"{tag}.npy", mmap_mode="r")[:, r0:r1:rs, c0:c1:cs])
        assert np.array_equal(np.isfinite(G[0]), valid), "ground nodata pattern differs from the static grid"
        Gflat = G[:, valid]
        Aflat = m.add_air(Gflat, ls, tau, model)
        A = np.full_like(G, np.nan)
        A[:, valid] = Aflat
        np.save(d / m.WORK_SUBDIR / "master" / f"{tag}.npy", A)
        outside = (Gflat < GP_LO) | (Gflat > GP_HI)
        row = {"ls_deg": ls, "tau": tau, "air_min_k": Aflat.min(), "air_p1_k": np.percentile(Aflat, 1),
               "air_p99_k": np.percentile(Aflat, 99), "air_max_k": Aflat.max(),
               "ground_min_k": Gflat.min(), "ground_max_k": Gflat.max(),
               "ground_outside_rover_range_frac": float(outside.mean()),
               "ground_below_frac": float((Gflat < GP_LO).mean()), "ground_above_frac": float((Gflat > GP_HI).mean()),
               "air_min_where_outside_k": float(Aflat[outside].min()) if outside.any() else np.nan,
               "air_max_where_outside_k": float(Aflat[outside].max()) if outside.any() else np.nan}
        for h in SPREAD_HOURS:
            j = int(np.flatnonzero(m.FRAME_HOURS == h)[0])
            lab = m.frame_label(h)
            row[f"air_sd_{lab}_k"] = float(Aflat[j].std())
            row[f"ground_sd_{lab}_k"] = float(Gflat[j].std())
            row[f"air_p1_p99_spread_{lab}_k"] = float(np.diff(np.percentile(Aflat[j], [1, 99]))[0])
        if color_range is not None:
            row.update(color_below_frac=float(np.mean(Aflat < color_range[0])),
                       color_above_frac=float(np.mean(Aflat > color_range[1])))
        if export:
            assert rs == cs == 1, "exports are checked on unstrided windows only"
            transform = m.map_transform(meta, window)
            files = m.export_ls(A, tag, d, transform, name="tair", color_range=color_range)
            row.update(m.check_exports(A, files, transform, rover_lonlat, (ROVER_200M[0] - r0, ROVER_200M[1] - c0),
                                       color_range=color_range))
            row.update({f"{f}_mb": m.folder_size_mb(p) for f, p in files.items()})
            rover_air[ls] = A[:, ROVER_200M[0] - r0, ROVER_200M[1] - c0].astype(np.float64)
        row["elapsed_s"] = time.time() - t0
        rows.append(row)
    if export:
        m.write_legend(d, color_range, "Air temperature at REMS height (K)")
    out = pd.DataFrame(rows).set_index("ls_deg")
    out.to_csv(d / "map_summary.csv")
    pd.options.display.float_format = "{:.3f}".format
    print(f"run {name}:")
    print(out.T.to_string())
    return out


results, rover_air = {}, {}
if MODE == "dry":
    results["sample"] = run_window("sample", DRY_RUNS["sample"][0], False, None)
    s = results["sample"]
    COLOR_RANGE = (float(np.floor(s.air_min_k.min() / COLOR_ROUND_K) * COLOR_ROUND_K),
                   float(np.ceil(s.air_max_k.max() / COLOR_ROUND_K) * COLOR_ROUND_K))
    print(f"crater sample: air {s.air_min_k.min():.1f}-{s.air_max_k.max():.1f} K -> proposed AIR_COLOR_RANGE_K = {COLOR_RANGE}")
else:
    COLOR_RANGE = AIR_COLOR_RANGE_K
for name, (window, export) in RUNS.items():
    if export:
        results[name] = run_window(name, window, True, COLOR_RANGE)

## Rover pixel against REMS (information, not a pass/fail check)

Map air at the rover's 200 m pixel against the mean REMS air temperature by LTST hour over the training sols within 15 deg of each map Ls (main rows). The map runs at one tau per Ls and at the 200 m pixel, so a few K of difference is expected.

In [ ]:
if rover_air:
    rems = tbl[~tbl.air_missing & ~tbl.tau_is_placeholder & ~tbl.sol.between(*m.SUSPECT_SOLS)].copy()
    rems["hour"] = rems.bin // 4
    comp = []
    for ls, a in rover_air.items():
        near = rems[np.abs(np.mod(rems.ls_deg - ls + 180.0, 360.0) - 180.0) <= COMPARE_LS_WINDOW_DEG]
        obs = near.groupby("hour").air_temp_k.mean()
        mod = pd.Series(a, index=m.FRAME_HOURS).groupby(np.floor(m.FRAME_HOURS).astype(int)).mean()
        diff = (mod - obs).dropna()
        comp.append({"ls_deg": ls, "n_sols": near.sol.nunique(), "mean_diff_k": diff.mean(),
                     "max_abs_diff_k": diff.abs().max(), "at_hour": int(diff.abs().idxmax())})
    comp = pd.DataFrame(comp).set_index("ls_deg")
    print(comp.to_string())

## Colour range (dry run) or manifest (full run)

In [ ]:
if MODE == "dry":
    print(f"dry run: proposed AIR_COLOR_RANGE_K = {COLOR_RANGE}; no manifest")
else:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT, check=True).stdout.strip()
    dirty = bool(subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=ROOT, check=True).stdout.strip())
    res = results["full"]
    manifest = {
        "title": "Gale Crater near-surface air temperature (REMS boom height), from the ground map and an air model",
        "created": time.strftime("%Y-%m-%d %H:%M:%S"), "git_commit": commit, "git_dirty": dirty, "mode": MODE,
        "grid": gman["grid"], "frames": gman["frames"], "scenarios": gman["scenarios"],
        "ground_map": {"manifest": "../ground/map_manifest.json", "git_commit": gman["git_commit"]},
        "encodings": {
            "A": {**gman["encodings"]["A"], "files": "cog/gale_tair_ls{LLL}.tif"},
            "B": gman["encodings"]["B"],
            "C": {"files": "png_color/ls{LLL}/ltst_{hhmm}.png", "legend": "png_color/legend.png", "colormap": m.COLOR_MAP,
                  "range_k": list(COLOR_RANGE), "levels": m.COLOR_LEVELS, "nodata": "transparent",
                  "clipped_fraction_below_above": {m.ls_tag(ls): [float(res.loc[ls, "color_below_frac"]),
                                                                  float(res.loc[ls, "color_above_frac"])] for ls, _ in LS_TAU}},
            "D": {**gman["encodings"]["D"], "files": "raw/gale_tair_ls{LLL}.u16"},
            "decode": gman["encodings"]["decode"]},
        "model": {
            "variant": m.AIR_VARIANT, "static_terms": model.static,
            "basis": "daily harmonics 1-3 and seasonal harmonics 1-2 of LTST and Ls, each x (1, tau); tau; "
                     "ground x (1, sin, cos of daily harmonic 1). Ridge alpha 1.0 on standardized columns. No clamp, no elevation.",
            "ground_input": "the ground map (../ground/_working/master/ls{LLL}.npy); in training, each REMS row's out-of-fold predicted ground "
                            "(flann_safe_no_ti), so training and map inputs carry the same kind of error",
            "fitting_rows": len(fit_frame), "lobo": LOBO,
            "decision": "fails the adoption rule (needs >= 0.2 K better than the benchmark macro and better in >= 5 of 9 blocks); "
                        "published under the fallback fixed on  28 as the better of the two configurations",
            "ground_pred_rover_range_k": [GP_LO, GP_HI],
            "ground_outside_rover_range_fraction": {m.ls_tag(ls): float(res.loc[ls, "ground_outside_rover_range_frac"]) for ls, _ in LS_TAU},
            "dair_dground_by_hour_csv": "outputs/model_scores/5_air_model/air_ground_sensitivity_by_hour.csv"},
        "caveats": [
            f"Does not beat the benchmark: LOBO macro {LOBO['air_hybrid_macro_k']} K vs {LOBO['benchmark_macro_k']} K for the "
            f"per-(LTST, Ls) mean of REMS air; better in {LOBO['blocks_better_than_benchmark']} of 9 blocks. Without block 0: "
            f"{LOBO['air_hybrid_macro_no_block0_k']} vs {LOBO['benchmark_macro_no_block0_k']} K.",
            "Night air is nearly the same everywhere at a given time. This is not physics: the model uses ground at about "
            "0.03-0.08 K per K at 00-05 h because block 0's predicted ground is 6.9 K too warm at night; with observed ground "
            "the REMS data shows about 0.2 K per K. By day, air moves about 0.5 K per K of ground.",
            "No elevation, lapse rate, cold-air pooling or slope (katabatic) flows: terrain enters only through the ground map. "
            "Real night air in a crater varies with these.",
            "Spatial skill is not measured: all rover positions lie within about 10 x 5 km.",
            "Ground outside the rover's range (1st-99th percentile of predicted ground) is extrapolated linearly into air; "
            "fractions per Ls in model.ground_outside_rover_range_fraction.",
            "Air is at REMS boom height (about 1.6 m). All ground-map caveats apply (../ground/map_manifest.json).",
        ],
    }
    (OUT / "map_manifest.json").write_text(json.dumps(manifest, indent=1))
    print("wrote", OUT / "map_manifest.json")

## Preview

In [ ]:
for name, (window, _) in RUNS.items():
    d = run_dir[name]
    fig, axes = plt.subplots(2, len(LS_TAU), figsize=(3.2 * len(LS_TAU), 6.2), squeeze=False)
    for k, (ls, tau) in enumerate(LS_TAU):
        A = np.load(d / m.WORK_SUBDIR / "master" / f"{m.ls_tag(ls)}.npy")
        for i, hour in enumerate(SPREAD_HOURS[::-1]):
            j = int(np.flatnonzero(m.FRAME_HOURS == hour)[0])
            im = axes[i, k].imshow(A[j], cmap=m.COLOR_MAP, vmin=COLOR_RANGE[0], vmax=COLOR_RANGE[1])
            axes[i, k].set_title(f"air, Ls {ls:g}, tau {tau:.2f}, {m.frame_label(hour)}", fontsize=8)
            axes[i, k].set_xticks([]); axes[i, k].set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.6, label="K")
    fig.savefig(d / "preview.png", dpi=110)
    plt.close(fig)
    print("saved", d / "preview.png")